In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\Aashish\AppData\Local\Temp\ipykernel_21076\1992512939.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# API Integration
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

print("API key loaded:", os.environ['GROQ_API_KEY'] is not None)

API key loaded: True


In [3]:
# NVIDIA LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3,
    max_tokens=1000,
)

# RAG prompt
prompt = ChatPromptTemplate.from_template("""
You are an agricultural assistant.

Use ONLY the information in the CONTEXT to answer the QUESTION.

Return only the final answer.
Do not explain your reasoning.
Do not discuss the context or instructions.

If the context does not contain enough information, respond exactly:
I don't have enough information in my knowledge base to answer that.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""")

In [4]:
print(llm.model)

openai/gpt-oss-120b


In [5]:
response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello! I hope you're having a wonderful day.


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"}
)

vectorstore = FAISS.load_local(
    "faiss_db",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

d:\rag chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 35540.33it/s]


In [7]:
chain = prompt | llm

def generate_answer(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine retrieved document contents
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Generate answer
    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response.content

In [8]:
query = "चावल की खेती कैसे करें in english"

answer = generate_answer(query)

print(answer)

**How to cultivate rice (in English)**  

1. **Land preparation**  
   - Plough the field thoroughly.  
   - Apply a basal dose of lime ≈ 140 kg per acre at the first ploughing.  
   - Incorporate organic manure ≈ 2 t per acre while ploughing.  

2. **Seed selection & treatment**  
   - **Direct‑sown rice:** use 32–40 kg of seed per acre.  
   - **Transplanted rice:** use 24–34 kg of seed per acre.  
   - Treat all seed with *Pseudomonas fluorescens* 10 g per kg seed (often with carbendazim or thiram for other crops).  

3. **Sowing method & timing**  
   - **Direct sowing:** sow in April before the rains start; wet seeding can be done after the first rain.  
   - **Transplanting:** raise a nursery from the treated seed and transplant seedlings later (usually after the rains begin).  

4. **Weed management**  
   - Apply a pre‑emergence herbicide 6–9 days after sowing for wet‑seeded rice (or 0–6 days for dry‑seeded rice):  
     - Butachlor 50 EC – 1 L per acre, **or**  
     - Pretila